In [ ]:
# This bot allows interactive querying of SQL tables, generating rapid 
# statistical summaries and dynamic visualizations to accelerate Exploratory Data Analysis (EDA).
import pyodbc
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

# SQL Server connection setup
server = 'MY_SERVER_NAME'
database = 'RetailRocketDB'
conn_str = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
engine = create_engine(conn_str, fast_executemany=True)

def get_table_summary(table_name):
    """Fetch table from SQL, print summary statistics, and generate appropriate visualizations."""
    try:
        # Load table into DataFrame
        df = pd.read_sql(f"SELECT * FROM dbo.{table_name}", engine)
        
        print(f"\n✅ Table successfully fetched: {table_name}")
        print(f"Total rows: {len(df)}")
        print("\nColumn Data Types:")
        print(df.dtypes)
        
        # Null value analysis
        nulls = df.isnull().sum()
        null_pct = (nulls / len(df) * 100).round(2)
        
        print("\n📊 Null Value Ratios (%):")
        print(null_pct[null_pct > 0])
        
        print("\n🔍 First 5 rows:")
        print(df.head())

        # ---------------------------
        # Smart / Dynamic Visualization
        # ---------------------------
        plt.figure(figsize=(10, 6))

        if "event_date" in df.columns and "events_per_day" in df.columns:
            # Date-based -> Bar chart
            df["event_date"] = pd.to_datetime(df["event_date"])
            sns.barplot(x="event_date", y="events_per_day", data=df, color="royalblue")
            plt.xticks(rotation=45)
            plt.title("Daily Event Counts")
            
            max_val = df["events_per_day"].max()
            max_date = df.loc[df["events_per_day"].idxmax(), "event_date"]
            print(f"📝 Insight: Peak activity observed on {max_date.date()} with {max_val} events.")

        elif "event" in df.columns and "event_count" in df.columns:
            # Category-based -> Bar chart
            sns.barplot(x="event", y="event_count", data=df, color="seagreen")
            plt.xticks(rotation=45)
            plt.title("Event Type Distribution")
            
            top_event = df.loc[df["event_count"].idxmax(), "event"]
            print(f"📝 Insight: Most frequent event type is '{top_event}'.")

        elif "event_hour" in df.columns and "events_per_hour" in df.columns:
            # Hour-based -> Bar chart
            sns.barplot(x="event_hour", y="events_per_hour", data=df, color="darkorange")
            plt.title("Hourly Event Distribution")

            peak_hour = df.loc[df["events_per_hour"].idxmax(), "event_hour"]
            print(f"📝 Insight: Peak activity hour is {peak_hour}:00.")

        elif "month" in df.columns and "events_per_month" in df.columns:
            # Month-based -> Bar chart
            sns.barplot(x="month", y="events_per_month", data=df, color="purple")
            plt.title("Monthly Event Distribution")

            peak_month = df.loc[df["events_per_month"].idxmax(), "month"]
            print(f"📝 Insight: Peak activity month is Month {peak_month}.")

        else:
            print("ℹ️ No specific visualization defined for this table schema.")

        plt.tight_layout()
        plt.show()
        
        return df
    
    except Exception as e:
        print(f"⚠️ Error: {e}")

# ---------------------------
# Interactive Bot Loop
# ---------------------------
while True:
    tablo = input("📂 Enter table name to analyze (or 'q' to quit) >>> ")
    if tablo.lower() == 'q':
        print("Shutting down bot... Goodbye! 👋")
        break
    else:
        df = get_table_summary(tablo)

# Automatically fetches and visualizes the requested SQL table based on user input
